# 01 — Camada Bronze | CineData Analytics

**Objetivo:** ingerir os CSVs brutos (Volume do Databricks) e a cotação do dólar (API do Banco Central) para tabelas **Delta** na camada Bronze.

**Regras de negócio aplicadas**
- A Bronze é um espelho dos CSVs originais: **nenhuma limpeza, renomeação ou conversão de tipo**. Por isso os CSVs são lidos com `inferSchema=False` (todas as colunas como `STRING`) — assim valores sujos como `"Unknown"`, `"$1,200"` ou textos deslocados não são perdidos nem alterados na leitura.
- Única coluna adicionada: `ingestion_datetime` (timestamp da ingestão).
- Gravação em **Delta** com modo **append** 
- Execuções repetidas do notebook geram novas linhas na Bronze — comportamento esperado do `append`.

## 1. Configuração

In [ ]:
import re
import requests
from datetime import date, datetime, timedelta

from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

# ---- Parâmetros do ambiente (ajuste aqui se o seu catálogo/volume for diferente) ----
CATALOG     = "workspace"
VOLUME_PATH = "/Volumes/workspace/default/Inputs/"
SCHEMA      = "bronze"

spark.sql(f"USE CATALOG {CATALOG}")
# Banco de dados (schema) da camada Bronze
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA}")

## 2. Ingestão dos CSVs

O mapeamento abaixo associa cada tabela Bronze a um padrão de nome de arquivo.

In [ ]:
display(dbutils.fs.ls(VOLUME_PATH))

In [ ]:
ARQUIVOS = {
    "bronze.tb_movies_info": f"{VOLUME_PATH}movies_info_TMDB_IMDB.csv",
    "bronze.tb_movies_financials": f"{VOLUME_PATH}movies_financials_IMDB_TMDB.csv",
    "bronze.tb_movies_metrics": f"{VOLUME_PATH}movies_metrics_IMDB_TMDB.csv",
    "bronze.tb_credits_and_tags": f"{VOLUME_PATH}credits_and_tags_IMDB_TMDB.csv",
    "bronze.tb_movies_reviews": f"{VOLUME_PATH}movies_reviews.csv",
}

def carregar_bronze(tabela: str, caminho: str) -> None:
    """Lê o CSV sem inferir tipos e grava na Bronze em Delta."""
    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", False)
        .option("multiLine", True)
        .option("quote", '"')
        .option("escape", '"')
        .csv(caminho)
    )

    df = df.withColumn("ingestion_datetime", F.current_timestamp())

    df.write.format("delta").mode("append").saveAsTable(tabela)

    print(f" {tabela} <- {caminho.split('/')[-1]}")


for tabela, caminho in ARQUIVOS.items():
    carregar_bronze(tabela, caminho)

## 3. Ingestão via API — Cotação do Dólar (Banco Central / PTAX)

 As datas são recebidas por **widgets** no formato `MM-DD-AAAA` (exigido pela API). Padrão: **últimos 7 dias corridos**, pois a API não retorna cotação em fins de semana/feriados.
 Se a API do Banco Central não puder ser acessada, o notebook usa **automaticamente** um histórico **mock** (fallback) e avisa no output (os valores foram obtidos previamente a partir do endpoint oficial do Banco Central).

In [ ]:
hoje = date.today()
dbutils.widgets.text("data_inicio", (hoje - timedelta(days=7)).strftime("%m-%d-%Y"), "Data inicial (MM-DD-AAAA)")
dbutils.widgets.text("data_fim",    hoje.strftime("%m-%d-%Y"),                      "Data final (MM-DD-AAAA)")

data_inicio = dbutils.widgets.get("data_inicio")
data_fim    = dbutils.widgets.get("data_fim")

# Valida o formato das datas (lança ValueError se estiver errado)
dt_ini = datetime.strptime(data_inicio, "%m-%d-%Y")
dt_fim = datetime.strptime(data_fim,    "%m-%d-%Y")
assert dt_ini <= dt_fim, "data_inicio deve ser <= data_fim"
print(f"Período consultado: {data_inicio} → {data_fim}")

In [ ]:
URL_BCB = (
    "https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
    "CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)"
)
PARAMS = {
    "@dataInicial":       f"'{data_inicio}'",
    "@dataFinalCotacao":  f"'{data_fim}'",
    "$select":            "dataHoraCotacao,cotacaoCompra",
    "$orderby":           "dataHoraCotacao asc",
    "$top":               1000,
    "$format":            "json",
}

# Histórico mock (fallback quando a API não está acessível)
mock_cotacoes = [
    {"dataHoraCotacao": "2026-10-01 13:10:35.4469",   "cotacaoCompra": 5.20730},
    {"dataHoraCotacao": "2026-10-02 13:03:16.256632", "cotacaoCompra": 5.22320},
    {"dataHoraCotacao": "2026-10-05 13:07:36.558602", "cotacaoCompra": 4.98530},
    {"dataHoraCotacao": "2026-10-06 13:03:21.267287", "cotacaoCompra": 4.96920},
]

try:
    resp = requests.get(URL_BCB, params=PARAMS, timeout=60)
    resp.raise_for_status()
    dados = resp.json()["value"]
    print(f"API do Banco Central OK — {len(dados)} registros.")
except requests.RequestException as erro:
    print("Não foi possível acessar a API do Banco Central.")
    print(f"Erro: {erro}")
    print("Usando histórico mock como fallback.")
    dados = mock_cotacoes

In [ ]:
schema_cotacao = StructType([
    StructField("dataHoraCotacao", StringType(), True),
    StructField("cotacaoCompra",   DoubleType(), True),
])

linhas = [(d.get("dataHoraCotacao"), None if d.get("cotacaoCompra") is None else float(d["cotacaoCompra"])) for d in dados]

if not linhas:
    print("A API não retornou cotações no período informado (ex.: só feriados/fins de semana). Nada gravado.")
else:
    (
        spark.createDataFrame(linhas, schema_cotacao)
        .withColumn("ingestion_datetime", F.current_timestamp())
        .write.format("delta").mode("append").saveAsTable("bronze.tb_cotacao_dolar")
    )
    print("bronze.tb_cotacao_dolar gravada.")

## 4. Validação da Bronze

In [ ]:
tabelas_bronze = list(ARQUIVOS.keys()) + ["bronze.tb_cotacao_dolar"]

for t in tabelas_bronze:
    formato = spark.sql(f"DESCRIBE DETAIL {t}").first()["format"]
    n = spark.table(t).count()
    print(f"{t:<35} formato={formato:<6} linhas={n}")

display(spark.table("bronze.tb_cotacao_dolar").orderBy("dataHoraCotacao"))